In [0]:
import pandas as pd
import matplotlib as plt

df = pd.read_csv('data.csv', sep=';')
df = df.dropna()
df = df.drop_duplicates()
df

In [0]:
import numpy as np

# Example: Categorize 'age' into bins and encode 'gender' as numeric
if 'age' in df.columns:
    df['age_category'] = pd.cut(df['age'], bins=[0, 18, 35, 50, 65, np.inf], labels=[0, 1, 2, 3, 4])
if 'gender' in df.columns:
    df['gender_numeric'] = df['gender'].map({'male': 0, 'female': 1})

# Stratify data by graduation status
if 'status' in df.columns:
    graduated_df = df[df['status'] == 'graduated']
    dropped_out_df = df[df['status'] == 'dropped out']
    enrolled_df = df[df['status'] == 'enrolled']

# Convert categorical columns to dummy variables for regression
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict and evaluate
y_rf_pred = rf_model.predict(X_test)
print(classification_report(y_test, y_rf_pred))

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Prepare target variable: graduated (1) vs dropped out (0)
if 'Target_Graduate' in df.columns:
    # Target column was already converted to dummies by Cell 2
    df['target'] = df['Target_Graduate'].astype(int)
else:
    raise ValueError("Target_Graduate column not found. Cell 2 may not have run successfully.")

# Select relevant features (exclude target and Target dummy columns)
feature_cols = [col for col in df.columns if col not in ['target', 'Target_Graduate', 'Target_Enrolled']]

X = df[feature_cols]
y = df['target']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

# Plot graduation status counts
if 'Target_Graduate' in df.columns:
    plt.figure(figsize=(6,4))
    sns.countplot(x='Target_Graduate', data=df)
    plt.title('Graduation Status Counts')
    plt.xlabel('Graduated')
    plt.ylabel('Count')
    plt.show()

# Plot ROC curve for logistic regression model
if 'y_test' in locals() and 'model' in locals():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8,6))
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.show()

    # Plot Precision-Recall curve for logistic regression model
    precision, recall, pr_thresholds = precision_recall_curve(y_test, y_prob)
    avg_precision = average_precision_score(y_test, y_prob)
    plt.figure(figsize=(8,6))
    plt.plot(recall, precision, color='green', lw=2, label=f'Precision-Recall curve (AP = {avg_precision:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc='lower left')
    plt.show()

In [0]:
import numpy as np

# For Logistic Regression: use absolute value of coefficients
if 'model' in locals() and hasattr(model, 'coef_'):
    lr_importances = np.abs(model.coef_[0])
    lr_top_indices = np.argsort(lr_importances)[::-1]
    print("Top contributing variables for Logistic Regression:")
    for idx in lr_top_indices[:10]:
        print(f"{feature_cols[idx]}: {lr_importances[idx]:.4f}")

# For Random Forest: use feature_importances_
if 'rf_model' in locals() and hasattr(rf_model, 'feature_importances_'):
    rf_importances = rf_model.feature_importances_
    rf_top_indices = np.argsort(rf_importances)[::-1]
    print("\nTop contributing variables for Random Forest:")
    for idx in rf_top_indices[:10]:
        print(f"{feature_cols[idx]}: {rf_importances[idx]:.4f}")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute confusion matrix for logistic regression model
if 'y_test' in locals() and 'y_pred' in locals():
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix (Logistic Regression)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()